In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

import joblib

print("Artham CATE environment ready!")


Artham CATE environment ready!


In [ ]:
df=pd.read_csv("artham_cate_candidates_90_categories_v2.csv")
print("dataset shape",df.shape)
print("dataset info",df.info())

dataset shape (2026, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2026 entries, 0 to 2025
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   example_id        2026 non-null   object
 1   description       2026 non-null   object
 2   input_source      2026 non-null   object
 3   seed_text         2026 non-null   object
 4   seed_type         2026 non-null   object
 5   parent_category   2026 non-null   object
 6   child_category    2026 non-null   object
 7   category_id       2026 non-null   object
 8   label_source      2026 non-null   object
 9   verified          2026 non-null   bool  
 10  review_status     2026 non-null   object
 11  ambiguity_flag    2026 non-null   object
 12  taxonomy_version  2026 non-null   object
 13  split_group       2026 non-null   object
 14  quality_status    2026 non-null   object
 15  quality_reason    2026 non-null   object
dtypes: bool(1), object(15)
memory usage

In [ ]:
x=df['description']
y=df['child_category']

print("input examples",len(x))
print("target labels",len(y))

print("Sample description:", x.iloc[0])
print("Correct category:", y.iloc[0])

input examples 2026
target labels 2026
Sample description: grocery purchase
Correct category: Groceries


# #Train/Test **Splititng**

In [ ]:

category_counts= y.value_counts()
category_counts

print("Total categories:", y.nunique())
print("Fewest examples in a category:", category_counts.min())
print("Categories with fewer than 2 examples:",
      (category_counts < 2).sum())


Total categories: 89
Fewest examples in a category: 9
Categories with fewer than 2 examples: 0


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training examples:", len(x_train))
print("Testing examples:", len(x_test))
print("Training categories:", y_train.nunique())
print("Testing categories:", y_test.nunique())

Training examples: 1620
Testing examples: 406
Training categories: 89
Testing categories: 89


In [ ]:
print("Categories before splitting:", y.nunique())

all_categories = set(df["category_id"])
train_categories = set(y_train)
test_categories = set(y_test)

print("Missing from training:", all_categories - train_categories)
print("Missing from testing:", all_categories - test_categories)

Categories before splitting: 89
Missing from training: {'entertainment.theme_parks', 'family_personal.donations', 'investments.fixed_deposit', 'food_dining.food_delivery', 'entertainment.events', 'family_personal.pet_care', 'investments.emergency_fund', 'entertainment.music', 'education.certifications', 'transport.vehicle_maintenance', 'bills_utilities.water', 'housing.rent', 'trips_travel.train_travel', 'shopping.home_decor', 'entertainment.movies', 'education.stationery', 'investments.stocks', 'housing.interior_works', 'health.lab_tests', 'education.books', 'transport.auto', 'family_personal.personal_care', 'education.school_fees', 'transport.parking_toll', 'investments.recurring_deposit', 'family_personal.salon', 'shopping.appliances', 'trips_travel.hotels', 'education.exam_fees', 'health.medicines', 'food_dining.restaurants', 'transport.vehicle_insurance', 'education.college_fees', 'bills_utilities.society_charges', 'health.wellness', 'education.courses', 'housing.security', 'trans

In [ ]:
print("Dataset categories:", df["category_id"].nunique())
print("Target categories:", y.nunique())

print("\nFirst 5 dataset labels:")
print(df["category_id"].head().tolist())

print("\nFirst 5 training labels:")
print(y_train.head().tolist())

print("\nFirst 5 testing labels:")
print(y_test.head().tolist())

print("\nTypes:")
print(type(y_train))
print(type(y_test))

Dataset categories: 90
Target categories: 89

First 5 dataset labels:
['food_dining.groceries', 'food_dining.groceries', 'food_dining.groceries', 'food_dining.groceries', 'food_dining.groceries']

First 5 training labels:
['Clothing', 'Stocks', 'Donations', 'School Fees', 'Internet']

First 5 testing labels:
['Movies', 'Vehicle Insurance', 'Doctor', 'Electronics', 'Repairs']

Types:
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [ ]:
from sklearn.model_selection import train_test_split

# Input: transaction descriptions
X = df["description"].astype(str)

# Target: unique parent.child category IDs
y = df["category_id"].astype(str)

# Recreate the split using corrected labels
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset categories:", y.nunique())
print("Training categories:", y_train.nunique())
print("Testing categories:", y_test.nunique())

print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

Dataset categories: 90
Training categories: 90
Testing categories: 89
Training examples: 1620
Testing examples: 406


In [ ]:
missing_categories = set(y_train.unique()) - set(y_test.unique())

print("Missing categories in test set:")
print(missing_categories)

print("\nOriginal counts:")
print(y.value_counts().loc[list(missing_categories)])

Missing categories in test set:
{'housing.furniture'}

Original counts:
category_id
housing.furniture    3
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF feature extractor
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Learn vocabulary from training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform testing data using the same vocabulary
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Training matrix: (1620, 2401)
Testing matrix: (406, 2401)
Vocabulary size: 2401


In [ ]:
from sklearn.linear_model import LogisticRegression

# Create our classification model
cate_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train using TF-IDF features and correct category labels
cate_model.fit(X_train_tfidf, y_train)

print("Artham CATE Baseline v0.1 trained successfully!")
print("Categories learned:", len(cate_model.classes_))

Artham CATE Baseline v0.1 trained successfully!
Categories learned: 90


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# Predict categories for unseen test descriptions
y_pred = cate_model.predict(X_test_tfidf)

# Calculate overall accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Artham CATE Accuracy: {accuracy:.2%}")

# Evaluate performance across categories
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


Artham CATE Accuracy: 97.04%

Classification Report:
                                 precision    recall  f1-score   support

            bills_utilities.dth       0.71      1.00      0.83         5
    bills_utilities.electricity       1.00      1.00      1.00         7
            bills_utilities.gas       0.67      1.00      0.80         4
       bills_utilities.internet       1.00      1.00      1.00         6
         bills_utilities.mobile       1.00      0.60      0.75         5
bills_utilities.society_charges       1.00      1.00      1.00         4
          bills_utilities.water       0.83      1.00      0.91         5
                education.books       1.00      1.00      1.00         5
       education.certifications       1.00      1.00      1.00         5
             education.coaching       1.00      1.00      1.00         5
         education.college_fees       1.00      1.00      1.00         3
              education.courses       1.00      1.00      1.00        

In [ ]:
import pandas as pd

errors = pd.DataFrame({
    "description": X_test.to_numpy(),
    "actual_category": y_test.to_numpy(),
    "predicted_category": y_pred
})

errors = errors[
    errors["actual_category"] != errors["predicted_category"]
]

print("Total incorrect predictions:", len(errors))

display(errors)

Total incorrect predictions: 12


,description,actual_category,predicted_category
5,nutrition purchase,health.supplements,food_dining.snacks_bakery
13,INR 450 debited towards nutrition,health.supplements,trips_travel.hotels
20,INR 450 debited towards air conditioner,shopping.appliances,trips_travel.flights
48,INR 450 debited towards spa,health.wellness,family_personal.salon
69,UPI PAYMENT TO dining,food_dining.restaurants,shopping.furniture
211,UPI PAYMENT TO water park,entertainment.theme_parks,bills_utilities.water
237,INR 450 debited towards airtel,bills_utilities.mobile,bills_utilities.dth
238,security service purchase,housing.security,transport.vehicle_maintenance
254,UPI PAYMENT TO nutrition,health.supplements,entertainment.movies
285,UPI PAYMENT TO HP,transport.fuel,bills_utilities.gas


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

cate_v02 = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            sublinear_tf=True
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            sublinear_tf=True
        ))
    ])),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

cate_v02.fit(X_train, y_train)

pred_v02 = cate_v02.predict(X_test)

print("CATE v0.2 accuracy:",
      accuracy_score(y_test, pred_v02))

print("CATE v0.2 macro F1:",
      f1_score(y_test, pred_v02,
               average="macro",
               zero_division=0))


CATE v0.2 accuracy: 0.9704433497536946
CATE v0.2 macro F1: 0.9614143035491352


In [ ]:
comparison = pd.DataFrame({
    "description": X_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "v01_prediction": y_pred,
    "v02_prediction": pred_v02
})

# Transactions where both models predicted differently
differences = comparison[
    comparison["v01_prediction"] != comparison["v02_prediction"]
]

print("Predictions that differ:", len(differences))
display(differences)

Predictions that differ: 7


,description,actual,v01_prediction,v02_prediction
5,nutrition purchase,health.supplements,food_dining.snacks_bakery,education.coaching
13,INR 450 debited towards nutrition,health.supplements,trips_travel.hotels,education.coaching
20,INR 450 debited towards air conditioner,shopping.appliances,trips_travel.flights,shopping.appliances
194,UPI PAYMENT TO book cab,transport.cab,transport.cab,education.books
211,UPI PAYMENT TO water park,entertainment.theme_parks,bills_utilities.water,entertainment.theme_parks
254,UPI PAYMENT TO nutrition,health.supplements,entertainment.movies,family_personal.donations
329,INR 450 debited towards book cab,transport.cab,transport.cab,education.books


In [ ]:
new_transactions = [
    "SWIGGY INSTAMART GROCERY ORDER",
    "AIRTEL XSTREAM FIBER BILL",
    "HPCL PETROL STATION",
    "NETFLIX MONTHLY SUBSCRIPTION",
    "AMAZON PURCHASE",
    "BOOK CAB TO AIRPORT",
    "APOLLO PHARMACY MEDICINES",
    "BESCOM ELECTRICITY BILL",
    "APPLE MUSIC SUBSCRIPTION",
    "HOTEL ROOM BOOKING"
]

# Convert descriptions using our existing vectorizer
new_features = vectorizer.transform(new_transactions)

# Predict categories
predictions = cate_model.predict(new_features)

# Display results
results = pd.DataFrame({
    "Transaction": new_transactions,
    "Predicted_Category": predictions
})

display(results)

,Transaction,Predicted_Category
0,SWIGGY INSTAMART GROCERY ORDER,food_dining.food_delivery
1,AIRTEL XSTREAM FIBER BILL,bills_utilities.internet
2,HPCL PETROL STATION,transport.fuel
3,NETFLIX MONTHLY SUBSCRIPTION,entertainment.subscriptions
4,AMAZON PURCHASE,shopping.online_shopping
5,BOOK CAB TO AIRPORT,trips_travel.cab_travel
6,APOLLO PHARMACY MEDICINES,health.medicines
7,BESCOM ELECTRICITY BILL,bills_utilities.electricity
8,APPLE MUSIC SUBSCRIPTION,entertainment.music
9,HOTEL ROOM BOOKING,trips_travel.hotels


In [ ]:

import numpy as np
import pandas as pd

# Obtain probabilities for all 90 categories.
probabilities = cate_model.predict_proba(new_features)

# Get the category names in the same order as probabilities.
categories = cate_model.classes_

# Sort category probabilities from highest to lowest.
top_indices = np.argsort(probabilities, axis=1)[:, ::-1]

# Highest-ranked category for every transaction.
first_indices = top_indices[:, 0]

# Second-highest-ranked category.
second_indices = top_indices[:, 1]

# Build a table showing the top two predictions.
confidence_results = pd.DataFrame({
    "Transaction": new_transactions,
    "First_Prediction": categories[first_indices],
    "First_Probability": np.round(
        probabilities[np.arange(len(new_transactions)), first_indices] * 100, 2
    ),
    "Second_Prediction": categories[second_indices],
    "Second_Probability": np.round(
        probabilities[np.arange(len(new_transactions)), second_indices] * 100, 2
    )
})

display(confidence_results)


,Transaction,First_Prediction,First_Probability,Second_Prediction,Second_Probability
0,SWIGGY INSTAMART GROCERY ORDER,food_dining.food_delivery,2.31,shopping.online_shopping,2.08
1,AIRTEL XSTREAM FIBER BILL,bills_utilities.internet,10.13,bills_utilities.electricity,2.72
2,HPCL PETROL STATION,transport.fuel,3.47,trips_travel.cab_travel,2.16
3,NETFLIX MONTHLY SUBSCRIPTION,entertainment.subscriptions,8.30,entertainment.ott,1.95
4,AMAZON PURCHASE,shopping.online_shopping,4.04,entertainment.ott,2.03
5,BOOK CAB TO AIRPORT,trips_travel.cab_travel,5.01,transport.cab,3.84
6,APOLLO PHARMACY MEDICINES,health.medicines,9.99,trips_travel.local_transport,1.55
7,BESCOM ELECTRICITY BILL,bills_utilities.electricity,10.83,health.hospitalization,1.60
8,APPLE MUSIC SUBSCRIPTION,entertainment.music,10.27,entertainment.subscriptions,2.78
9,HOTEL ROOM BOOKING,trips_travel.hotels,7.77,trips_travel.cab_travel,2.45


In [ ]:

# ============================================================
# ARTHAM CATE v0.1
# STEP 9A: VOCABULARY INSPECTION
# ============================================================

# STEP 1
# Retrieve the vocabulary learned during training.
#
# A vocabulary is the collection of words and phrases
# that our TF-IDF vectorizer knows.

vocabulary = vectorizer.vocabulary_


# STEP 2
# Select important terms from our independent tests.
#
# We'll check both merchant names and contextual words.

test_words = [
    "swiggy",
    "instamart",
    "grocery",
    "netflix",
    "subscription",
    "hpcl",
    "petrol",
    "amazon",
    "airtel",
    "xstream",
    "fiber",
    "bescom",
    "electricity",
    "pharmacy",
    "airport"
]


# STEP 3
# Check whether each word exists in the vocabulary.

for word in test_words:

    # The "in" operator checks vocabulary membership.
    if word in vocabulary:
        print(f"{word}: KNOWN")

    else:
        print(f"{word}: UNKNOWN")


swiggy: KNOWN
instamart: UNKNOWN
grocery: KNOWN
netflix: KNOWN
subscription: KNOWN
hpcl: KNOWN
petrol: KNOWN
amazon: KNOWN
airtel: KNOWN
xstream: KNOWN
fiber: KNOWN
bescom: KNOWN
electricity: KNOWN
pharmacy: KNOWN
airport: KNOWN


In [ ]:

# ============================================================
# ARTHAM CATE v0.1
# STEP 9B: TRANSACTION CONTEXT EXPERIMENT
# ============================================================

import pandas as pd

# STEP 1: Create different versions of a transaction.
#
# Each description changes one or more contextual clues.
# We want to see how the model responds.

test_descriptions = [
    "SWIGGY",
    "INSTAMART",
    "GROCERY",
    "SWIGGY GROCERY",
    "SWIGGY INSTAMART",
    "SWIGGY INSTAMART GROCERY ORDER",
    "GROCERY STORE PURCHASE",
    "FOOD DELIVERY FROM SWIGGY",
    "NETFLIX",
    "NETFLIX SUBSCRIPTION",
    "OTT SUBSCRIPTION NETFLIX"
]


# STEP 2: Convert the text into TF-IDF features.
#
# We reuse the vectorizer trained earlier.
# transform() does not change its vocabulary.

test_features = vectorizer.transform(test_descriptions)


# STEP 3: Predict a category for each description.
#
# cate_model is our existing CATE v0.1 classifier.

test_predictions = cate_model.predict(test_features)


# STEP 4: Get the model's top probability.
#
# These probabilities are uncalibrated, so they
# should not be treated as verified confidence levels.

test_probabilities = cate_model.predict_proba(test_features)

top_probabilities = test_probabilities.max(axis=1) * 100


# STEP 5: Present predictions in a readable table.

context_results = pd.DataFrame({
    "Transaction": test_descriptions,
    "Predicted_Category": test_predictions,
    "Top_Probability_%": top_probabilities.round(2)
})

display(context_results)


,Transaction,Predicted_Category,Top_Probability_%
0,SWIGGY,food_dining.food_delivery,3.99
1,INSTAMART,trips_travel.local_transport,1.70
2,GROCERY,food_dining.groceries,2.66
3,SWIGGY GROCERY,food_dining.food_delivery,2.72
4,SWIGGY INSTAMART,food_dining.food_delivery,3.99
5,SWIGGY INSTAMART GROCERY ORDER,food_dining.food_delivery,2.31
6,GROCERY STORE PURCHASE,food_dining.groceries,2.63
7,FOOD DELIVERY FROM SWIGGY,food_dining.food_delivery,6.10
8,NETFLIX,entertainment.ott,3.16
9,NETFLIX SUBSCRIPTION,entertainment.subscriptions,4.51


In [ ]:

# ============================================================
# ARTHAM CATE — STEP 10
# TRAINING DATA CONTEXT AUDIT
# ============================================================

import pandas as pd

# Step 1: Reconstruct the training data using
# the same indices from our original split.
#
# IMPORTANT:
# We inspect X_train and y_train, not X_test,
# so we avoid using test examples to modify training.

training_audit = pd.DataFrame({
    "description": X_train.to_numpy(),
    "category_id": y_train.to_numpy()
})


# Step 2: Define the keywords we want to investigate.
#
# These keywords caused interesting predictions
# during our independent transaction tests.

keywords = [
    "swiggy",
    "instamart",
    "grocery",
    "netflix",
    "subscription"
]


# Step 3: Search the training descriptions
# for each keyword, ignoring uppercase/lowercase.

for keyword in keywords:

    matches = training_audit[
        training_audit["description"].str.contains(
            keyword,
            case=False,
            regex=False,
            na=False
        )
    ]

    print("\n" + "=" * 55)
    print("KEYWORD:", keyword.upper())
    print("Matching training examples:", len(matches))

    # Step 4: Show the category distribution.
    #
    # This tells us which categories are associated
    # with the keyword in the training dataset.

    print("\nCategory distribution:")
    print(matches["category_id"].value_counts())

    # Step 5: Show example descriptions.
    #
    # head(10) displays up to 10 matching rows,
    # keeping our output manageable.

    print("\nSample training descriptions:")
    display(matches.head(10))



KEYWORD: SWIGGY
Matching training examples: 3

Category distribution:
category_id
food_dining.food_delivery    3
Name: count, dtype: int64

Sample training descriptions:


,description,category_id
15,INR 450 debited towards Swiggy,food_dining.food_delivery
686,UPI PAYMENT TO Swiggy,food_dining.food_delivery
757,Swiggy purchase,food_dining.food_delivery



KEYWORD: INSTAMART
Matching training examples: 0

Category distribution:
Series([], Name: count, dtype: int64)

Sample training descriptions:


,description,category_id



KEYWORD: GROCERY
Matching training examples: 2

Category distribution:
category_id
food_dining.groceries    2
Name: count, dtype: int64

Sample training descriptions:


,description,category_id
1081,INR 450 debited towards grocery,food_dining.groceries
1531,UPI PAYMENT TO grocery,food_dining.groceries



KEYWORD: NETFLIX
Matching training examples: 2

Category distribution:
category_id
entertainment.ott    2
Name: count, dtype: int64

Sample training descriptions:


,description,category_id
233,INR 450 debited towards netflix,entertainment.ott
851,netflix purchase,entertainment.ott



KEYWORD: SUBSCRIPTION
Matching training examples: 2

Category distribution:
category_id
entertainment.subscriptions    2
Name: count, dtype: int64

Sample training descriptions:


,description,category_id
163,subscription purchase,entertainment.subscriptions
737,UPI PAYMENT TO subscription,entertainment.subscriptions


In [ ]:

import joblib
from sklearn.pipeline import Pipeline
from google.colab import files

# Combine our trained TF-IDF vectorizer
# and Logistic Regression classifier.
cate_pipeline = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", cate_model)
])

# Save the complete model.
joblib.dump(cate_pipeline, "artham_cate_v01.joblib")

# Reload it to verify the saved model works.
loaded_model = joblib.load("artham_cate_v01.joblib")

print("Prediction:",
      loaded_model.predict(["HPCL PETROL STATION"])[0])

print("Categories:",
      len(loaded_model.classes_))

# Download the model to your Mac.
files.download("artham_cate_v01.joblib")


Prediction: transport.fuel
Categories: 90


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys
import sklearn
import joblib

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.16
scikit-learn: 1.6.1
joblib: 1.6.0
